In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import os
from datetime import datetime
from tqdm.auto import tqdm

In [ ]:
import quick_eval as qeval

In [ ]:
dir_lists = {
    'mimic': ['mimic_cn_conditions_2', 'mimic_cn_2'], 
    # 'ptbxl': ['ptbxl_cn', 'ptbxl_cn_after_mimic_cn']
}
# dir_lists = {
#     'mimic': ['mimic_cn_conditions', 'mimic_cn', 'mimic_cn_sch_with_restart'], 
#     'ptbxl': ['ptbxl_cn', 'ptbxl_cn_after_mimic_cn']
# }

In [ ]:
save_file = f"results_{datetime.now().strftime('%m%d%H%M%S')}.csv"
# save_file = f"results.csv"

In [ ]:
print(f"Saving results to {save_file}")
df_list = []
df = None
for data_type in tqdm(dir_lists.keys(), desc="Data Types"):
    for dir_name in tqdm(dir_lists[data_type], desc="Directories"):
        dir_path = f"/home/kumargirish/output_sssd-ecg/{dir_name}/{dir_name}/ch256_T200_betaT0.02/"
        sub_dir_list = []
        synth_dir_list = []
        
        for x in os.listdir(dir_path):
            if not x.endswith('plots'):
                y = os.path.join(dir_path, x)
                if (os.path.isdir(y) and os.listdir(y)):
                    sub_dir_list.append(x)
                    synth_dir_list.append(y)
                    
        results = qeval.multi_eval_with_dir_paths(
            dir_paths=synth_dir_list,
            data_type=data_type
        )
        
        for sub_dir, synth_dir, res in zip(sub_dir_list, synth_dir_list, results):
            if res:            
                res = {k: v['aggregated'] for k, v in res.items()}
                info_list = sub_dir.split('_')
                res['n_samples'] = int(info_list[-1])
                res['checkpoint_iter'] = int(info_list[-3]) 
                res['error'] = False
            else:
                res = {'error': True}
            
            res['dir_name'] = dir_name
            res['sub_dir'] = sub_dir
            
            df_list.append(res)
    
        df = pd.DataFrame(df_list)
        df.to_csv(save_file, index=False)

In [ ]:
df

In [ ]:
temp = df[df['error']==False].sort_values(by=['dir_name', 'checkpoint_iter']).reset_index(drop=True).drop(columns=['error'])
temp

In [ ]:
for _, r in temp.iterrows():
    print(r.to_dict())